In [1]:
import sys
import pandas as pd
import gurobipy as gp
import json

from pathlib import Path
import sys

try:
    ROOT = Path(__file__).resolve().parents[1]
except NameError:
    ROOT = Path.cwd().parent

sys.path.append(str(ROOT / "src"))

RESULTS = ROOT / "experiments" / "technology diffusion"
RESULTS.mkdir(parents=True, exist_ok=True)

from technology_diffusion import *

In [ ]:
c_list = [1, 5, 10, 20]
n_list = [200, 400, 600, 1000, 2000]
seed_list = [1, 42, 53, 99, 101]

In [ ]:
init_nodes = 5
init_mode = "complete"

delta = 0.5
xi = 0.1
d = 2
buffer_dim = 5000
min_conn = 20
mg_max_depth = 5
mg_memory_len = 10
verbose = 1

strategy = [degree_threshold, high_thetas_start, SD_start, degree_connected, degree, betweenness, random_start]

save_results_csv = True
results_csv_path = RESULTS / "technology_diffusion_results.csv"
save_gurobi_log = True
gurobi_log_path = RESULTS / "gurobi.log"

gurobi_env = gp.Env(empty=True)
gurobi_env.setParam("OutputFlag", 1 if save_gurobi_log else 0)
gurobi_env.setParam("LogToConsole", 0)
gurobi_env.setParam("LogFile", str(gurobi_log_path) if save_gurobi_log else "")
gurobi_env.start()

combinations = list(itertools.product(n_list, c_list, seed_list))
print(f"Total runs: {len(combinations)}")

results = []
start_all = time.time()

for run_idx, combo in enumerate(combinations, start=1):
    n_nodes, c, seed = combo
    max_time = 180 * (n_nodes//100)

    print("\n" + "=" * 90)
    print(
        f"Run {run_idx}/{len(combinations)} | "
        f"N={n_nodes}, c={c}, seed={seed}, init_nodes={init_nodes}, init_mode={init_mode}"
    )

    g, thetas = create_pa_graph(
        n_nodes=n_nodes,
        c=c,
        seed=seed,
        init_nodes=init_nodes,
        init_mode=init_mode,
    )

    print(f"Building Goldberg-Liu IP...", end='')
    start = time.time()
    model, x = build_golberg_liu_ip(g, thetas, max_time, env=gurobi_env)
    build_time = time.time() - start

    if max_time - build_time < 0:
        print(f'\rGoldberg-Liu IP build failed: build time {round(build_time, 2)}s exceeds max time {max_time}s.' + ' ' * 20)
        gl_build_failed = True
        gl_build_time = None
        gl_opt_time = None
        gl_runtime = None
        gl_k = None

    else:
        print(f'\rGoldberg-Liu IP built in {round(build_time, 2)} seconds!' + ' ' * 20)
        print(f"Solving Goldberg-Liu IP with time limit {round(max_time-build_time, 2)} seconds...", end='')
        suppress_print()
        model.setParam("TimeLimit", max_time - build_time)
        resume_print()
        model.optimize()
        print(f'\rGoldberg-Liu IP solved in {round(model.Runtime, 2)} seconds!' + ' ' * 20)

        gl_build_failed = False
        gl_build_time = round(build_time, 4)
        gl_opt_time = round(float(model.Runtime), 4)
        gl_runtime = round(build_time + gl_opt_time, 4)
        gl_k = int(round(model.objVal)) if model.SolCount > 0 else None

    ns_k, final_x, ns_runtime, history = NS_technology_diffusion_binary_search(
        g,
        thetas,
        strategy,
        delta,
        xi,
        d,
        min_conn,
        mg_max_depth,
        mg_memory_len,
        max_time,
        buffer_dim,
        verbose
    )

    history_json = json.dumps([list(event) for event in history])

    gap = (ns_k - gl_k) if (ns_k is not None and gl_k is not None) else None

    results.append([
        n_nodes,
        c,
        seed,
        g.number_of_nodes(),
        g.number_of_edges(),
        max_time,
        gl_build_failed,
        gl_build_time,
        gl_opt_time,
        gl_runtime,
        gl_k,
        ns_runtime,
        ns_k,
        gap,
        history_json,
    ])

    if gl_build_failed:
        print(f"Goldberg-Liu -> Build failed.")
    else:
        print(f"Goldberg-Liu -> K={gl_k}, runtime={round(gl_runtime, 2)}s")
    print(f"NS Binary    -> K={ns_k}, runtime={round(float(ns_runtime), 2)}s, " f"gap(NS-GL)={gap}")

elapsed_all = time.time() - start_all
gurobi_env.dispose()

columns = [
    "n_nodes",
    "c",
    "seed",
    "num_nodes",
    "num_edges",
    "max_time",
    "GL_build_failed",
    "GL_build_time_s",
    "GL_optimization_time_s",
    "GL_runtime_s",
    "GL_K",
    "NS_runtime_s",
    "NS_K",
    "NS_GL_gap",
    "NS_history_json",
]

results_df = pd.DataFrame(results, columns=columns)
results_df = results_df.sort_values(["n_nodes", "c", "seed"]).reset_index(drop=True)

print("\n" + "=" * 90)
print(f"Completed {len(results_df)} runs in {round(elapsed_all, 2)} seconds.")
display(results_df)

if save_results_csv:
    results_df.to_csv(results_csv_path, index=False)
    print(f"Results saved to: {results_csv_path}")

if save_gurobi_log:
    print(f"Gurobi log saved to: {gurobi_log_path}")

Total runs: 2

Run 1/2 | N=200, c=1, seed=1, init_nodes=5, init_mode=complete
Goldberg-Liu IP built in 4.04 seconds!                    
Goldberg-Liu IP solved in 6.04 seconds!                    
Binary search... Done! k :17. Time: 12/10 s.                                
Goldberg-Liu -> K=102, runtime=10.09s
NS Binary    -> K=17, runtime=11.94s, gap(NS-GL)=-85

Run 2/2 | N=200, c=1, seed=42, init_nodes=5, init_mode=complete
Goldberg-Liu IP built in 4.02 seconds!                    
Goldberg-Liu IP solved in 6.04 seconds!                    
Binary search... Done! k :16. Time: 10/10 s.                                
Goldberg-Liu -> K=105, runtime=10.06s
NS Binary    -> K=17, runtime=10.02s, gap(NS-GL)=-88

Completed 2 runs in 42.13 seconds.


,n_nodes,c,seed,num_nodes,num_edges,max_time,GL_build_failed,GL_build_time_s,GL_optimization_time_s,GL_runtime_s,GL_K,NS_runtime_s,NS_K,NS_GL_gap,NS_history_json
0,200,1,1,200,506,10,False,4.0416,6.044,10.0856,102,11.9406,17,-85,"[[200, 0.0], [100, 0.0011], [50, 0.0023], [25,..."
1,200,1,42,200,514,10,False,4.0179,6.040,10.0579,105,10.0166,17,-88,"[[200, 0.0], [100, 0.0008], [50, 0.0019], [25,..."


Results saved to: /Users/matteobergamaschi/Desktop/td_git/experiments/technology diffusion/technology_diffusion_results.csv
Gurobi log saved to: /Users/matteobergamaschi/Desktop/td_git/experiments/technology diffusion/gurobi.log


In [ ]:
static_params_path = RESULTS / "technology_diffusion_static_params.json"
static_params = {
    "init_nodes": init_nodes,
    "init_mode": init_mode,
    "delta": delta,
    "xi": xi,
    "d": d,
    "buffer_dim": buffer_dim,
    "min_conn": min_conn,
    "mg_max_depth": mg_max_depth,
    "mg_memory_len": mg_memory_len,
    "verbose": verbose,
    "strategy_names": [fn.__name__ for fn in strategy],
}

with open(static_params_path, "w", encoding="utf-8") as f:
    json.dump(static_params, f, indent=2)

print(f"Static parameters saved to: {static_params_path}")

Static parameters saved to: /Users/matteobergamaschi/Desktop/td_git/experiments/technology diffusion/technology_diffusion_static_params.json
